In [ ]:
import numpy as np
from pynq import Overlay, allocate
from pynq.lib import AXI_Lite
import soundfile as sf
import time

# 1. 加载 Overlay（等队友生成后，把 .bit 和 .hwh 放进 board/overlay/）
# 注意：路径是在板子的 Linux 系统里，不是 Windows 电脑！
overlay = Overlay("/home/xilinx/jupyter_notebooks/board/overlay/audio_accel.bit")
print("✅ Overlay 加载成功！")

# 2. 读取测试音频（板子上的路径）
audio_path = "/home/xilinx/jupyter_notebooks/data/audio/real_voice.wav"
data, fs = sf.read(audio_path)
if len(data.shape) > 1:
    x = data[:, 0] + data[:, 1]
else:
    x = data.copy()
x = x - np.mean(x)
print(f"音频读取成功，长度：{len(x)/fs:.2f} 秒")

# 3. 分配 DMA 缓冲区（大小根据队友的 IP 寄存器设定）
# 这里是一个占位示例，后续需要根据技术队友的寄存器映射来改
input_buffer = allocate(shape=(len(x),), dtype=np.float32)
output_buffer = allocate(shape=(len(x),), dtype=np.float32)
np.copyto(input_buffer, x)

# 4. 调用硬件加速核（使用 AXI-Lite 写寄存器启动）
# 注意：具体的寄存器地址和启动位需要队友提供
start_time = time.time()

# 伪代码示例：
# overlay.audio_accel.write(0x00, 0x01)  # 启动加速核
# while overlay.audio_accel.read(0x00) & 0x02 == 0:  # 等待完成
#     pass

end_time = time.time()
print(f"硬件加速处理耗时：{end_time - start_time:.4f} 秒")

# 5. 将硬件输出结果与 Python 基线对比
# 从 output_buffer 读取结果并保存，便于后续计算误差
# np.save("/home/xilinx/jupyter_notebooks/data/results/hw_output.npy", output_buffer)

print("✅ 测试完成，硬件结果已准备好与 Python 基线比对！")